# GPU Architecture

We start, as usual, by loading our ice magic.

In [1]:
%load_ext ice.magic

Using temporary directory /home/arslan


## FMA Benchmark

In the previous sections, we have used a given execution configuration for our kernels.
You might have wondered how to choose a suitable number of threads per block and blocks per grid.

To investigate the effect of different execution configurations, we switch to a different type of workload.
Instead of doing very little computation and moving a lot of data, we focus on doing a high number of **fused multiply-add (FMA)** operations.
Doing so allows us to shift the bottleneck from the shared global memory to the compute units, and thereby make the effects of different execution configurations more visible.

Check out and run the code below.
It performs one million FMA operations for each 'work item', while the work items are distributed across the available threads using a grid-stride loop.
In the current state, the number of work items is set to the total number of threads, which means that each thread performs a fixed number of one million FMA operations.
After execution, the benchmark reports an estimate of the achieved performance in TFLOP/s (trillion floating point operations per second).

In [2]:
%%cuda -c code/architecture/fma.cu

constexpr size_t numFMA = 1024 * 1024;

__global__ void fma(float* data, size_t numElements) {
    for (size_t i0 = blockIdx.x * blockDim.x + threadIdx.x; i0 < numElements; i0 += blockDim.x * gridDim.x) {
        float acc = i0;

        for (auto r = 0; r < numFMA; ++r)
            acc = 0.12f * acc + 1.2f;

        //# dummy check to prevent compiler from eliminating loop
        if (0.0 == acc)
            data[i0] = acc;
    }
}

int main(int argc, char *argv[]) {
    auto numBlocks = 84 * 0 + 1;
    auto numThreadsPerBlock = 64;
    auto numElements = numBlocks * numThreadsPerBlock;
    auto numIterations = 10;

    float *d_data;
    checkCudaError(cudaMalloc(&d_data, numElements * sizeof(float)));

    //# warm-up
    fma<<<numBlocks, numThreadsPerBlock>>>(d_data, numElements);

    cudaDeviceSynchronize();
    auto start = std::chrono::steady_clock::now();

    //# main 'work'
    for (auto it = 0; it < numIterations; ++it) {
        fma<<<numBlocks, numThreadsPerBlock>>>(d_data, numElements);
    }

    cudaDeviceSynchronize();
    auto end = std::chrono::steady_clock::now();

    const std::chrono::duration<double> elapsedSeconds = end - start;
    auto numFlopsPerElement = 2 * numFMA;
    std::cout << "Time elapsed:          " << elapsedSeconds.count() << " s\n";
    std::cout << "Estimated performance: " << 1e-12 * numFlopsPerElement * numElements * numIterations / elapsedSeconds.count() << " TFLOP/s\n";

    cudaFree(d_data);
}

FileNotFoundError: [Errno 2] No such file or directory: 'nvcc'

## Exercise: Observe Performance for Different Execution Configurations

Run the above example and observe the performance for different execution configurations.
Focus on the parameters given in the following table.
We are interested in:

* What is the dependency of performance on the number of threads for **a single block**?
* What is the dependency of performance on the number of threads for **a small number of blocks**?
* What is the dependency of performance on the number of threads for **a number of blocks around multiples of 84**?
    * We will discuss where this number comes from later.

You can use the following table to record your observations:

| Number of Blocks | Number of Threads per Block | Performance (TFLOPS) |
|------------------|-----------------------------|----------------------|
| 1                | 64                          |                      |
| 1                | 128                         |                      |
| 1                | 256                         |                      |
| 2                | 256                         |                      |
| 4                | 256                         |                      |
| 8                | 256                         |                      |
| 83               | 256                         |                      |
| 84               | 256                         |                      |
| 85               | 256                         |                      |
| 167              | 256                         |                      |
| 168              | 256                         |                      |
| 169              | 256                         |                      |
| 336              | 256                         |                      |
| 337              | 256                         |                      |
| 768              | 256                         |                      |

### Possible Solution

| Number of Blocks | Number of Threads per Block | Performance (TFLOPS) |
|------------------|-----------------------------|----------------------|
| 1                | 64                          | 0.05                 |
| 1                | 128                         | 0.1                  |
| 1                | 256                         | 0.2                  |
| 2                | 256                         | 0.4                  |
| 4                | 256                         | 0.9                  |
| 8                | 256                         | 1.7                  |
| 83               | 256                         | 18.0                 |
| 84               | 256                         | 18.2                 |
| 85               | 256                         | 18.0                 |
| 167              | 256                         | 35.5                 |
| 168              | 256                         | 35.6                 |
| 169              | 256                         | 18.4                 |
| 336              | 256                         | 35.8                 |
| 337              | 256                         | 26.9                 |
| 768              | 256                         | 35.8                 |

## Example Architecture - H100

Let's briefly look at the architecture of a modern Nvidia GPU, in this case an H100, to better understand what happened in the previous exercise.

<img align="right" src="img/h100-chip.png" alt="H100 Chip" width="384px"/>

A key design principle of all GPUs is their **hierarchical structure**, which is essential for achieving high levels of parallelism and scalability.
To understand GPU performance, we begin by examining the **NVIDIA H100 architecture**, a state-of-the-art example of modern GPUs.
The H100 demonstrates the hierarchical design and advanced capabilities that drive GPU performance.

For further details, refer to the official [NVIDIA H100 White Paper](https://resources.nvidia.com/en-us-hopper-architecture/nvidia-h100-tensor-c).
The following figures and information are derived from this resource as well.

<img align="right" src="img/h100-layout-annotated.png" alt="H100 Chip Layout with Annotations" width="768px" style="background-color:white"/>

The diagram on the right illustrates a 'full configuration' of one H100 chip, emphasizing the hierarchical arrangement of its components:
* **Graphics Processing Clusters (GPCs)** with multiple
* **Texture Processing Clusters (TPCs)** with multiple
* **Streaming Multiprocessors (SMs)** with multiple
* **SM Sub-Partitions (SMSPs)** (see below).

In practice, not all units in the full configuration are available. For example, the SXM5 version of the H100 includes:
* 8 GPCs, each with
* 8 or 9 TPCs, each with
* 2 SMs, each with
* 4 SMSPs,
for a total of **132 Streaming Multiprocessors (SMs)** available for computation.

<img align="right" src="img/h100-sm-layout.png" alt="H100 SM Layout" width="384px" style="background-color:white"/>

Each SM is further subdivided into 4 sub partitions, as visualized in the figure on the right, each with
* 16 INT32 units, for a total of 132 * 4 * 16 = 8448,
* 32 FP32 units, for a total of 132 * 4 * 32 = 16896,
* 16 FP64 units, for a total of 132 * 4 * 16 = 8448, and
* 1 tensor core, for a total of 132 * 4 = 528.
Each unit is capable of executing one fused-multiply-add (FMA) operation per cycle, with the exception of the tensor cores which can each perform 512 FP16/FP32-mixed-precision FMAs at once.

## Mapping Work to the GPU

When running a kernel, the threads are organized into a hierarchy of **grids**, **blocks**, (**warps**), and **threads**.
This hierarchy is designed to match the hierarchical structure of the GPU hardware.
In practice:

* A kernel is launched with a one grid containing one or more blocks.
* For each block in the current grid:
    * The GPU locates an SM with free capacity and schedules the block to that SM.
      The block will run on exactly one SM, and will not migrate to another SM during execution.
      Each SM can handle multiple blocks at the same time (constrains apply).
    * For each warp in the current block:
        * The GPU schedules the warp to an SMSP.
        * For each thread in the current warp:
            * The thread executes its instructions.

This explains a dependency of performance on the number of blocks in relation to the number of SMs.
As you might have already guessed, the number of SMs on an A40 GPU (which we use in this course) is 84.
In practice, there are multiple ways of querying the number of SMs.

First, looking up the relevant information in available documentation can be helpful, e.g.
* The [official data sheet](https://images.nvidia.com/content/Solutions/data-center/a40/nvidia-a40-datasheet.pdf), or
* third-party resources like [TechPowerUp](https://www.techpowerup.com/gpu-specs/a40-pcie.c3700).

Next, if access to the GPU in question is available, using `nvaccelinfo` (available via the [Nvidia HPC SDK](https://developer.nvidia.com/hpc-sdk)) provides detailed information about the GPU, including the number of SMs.

In [ ]:
!nvaccelinfo | grep "Number of Multiprocessors"

Finally, querying the number of SMs programmatically using the CUDA API is also possible.

```c++
cudaGetDeviceProperties(ptr_to_prop_struct, device)
```
* Queries hardware properties of an installed GPU.
* All values are stored in a provided `cudaDeviceProp` struct ([documentation](https://docs.nvidia.com/cuda/cuda-runtime-api/structcudaDeviceProp.html#structcudaDeviceProp)).
* [Documentation](https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__DEVICE.html#group__CUDART__DEVICE_1g1bf9d625a931d657e08db2b4391170f0)

In [ ]:
%%cuda -c code/architecture/device-props.cu

int main() {
    int device;
    cudaGetDevice(&device);

    cudaDeviceProp prop{};
    cudaGetDeviceProperties(&prop, device);

    std::cout << "\nDevice " << device << ": " << prop.name << "\n";
    std::cout << "  Compute Capability: " << prop.major << "." << prop.minor << "\n";
    std::cout << "  SMs: " << prop.multiProcessorCount << "\n";
    std::cout << "  Global Memory: " << std::fixed << std::setprecision(2)
              << (prop.totalGlobalMem / (1024.0 * 1024.0 * 1024.0)) << " GB\n";
    std::cout << "  Warp Size: " << prop.warpSize << "\n";
    std::cout << "  Max Threads per Block: " << prop.maxThreadsPerBlock << "\n";
    std::cout << "  Max Threads Dim: [" << prop.maxThreadsDim[0] << ", "
              << prop.maxThreadsDim[1] << ", " << prop.maxThreadsDim[2] << "]\n";
    std::cout << "  Max Grid Size: [" << prop.maxGridSize[0] << ", "
              << prop.maxGridSize[1] << ", " << prop.maxGridSize[2] << "]\n";

    return 0;
}

## Hardware-Aware Grid Design

By querying hardware properties like the number of streaming multiprocessors at runtime, the grid dimensions can be scaled according to the hardware capabilities.
This way, applications can effectively utilize smaller and larger GPUs well, provided that the workload itself is sufficient, without manual code changes. Use the following approach to design (performance-) portable grid configurations:

```c++
int main() {
    int deviceId;
    int numberOfSMs;
    int warpSize;

    auto numBlocks;
    auto numThreadsPerBlock;

    cudaGetDevice(&deviceId);
    
    // query the number of streaming multiprocessors
    cudaDeviceGetAttribute(&numberOfSMs, cudaDevAttrMultiProcessorCount, deviceId);

    // query the warp size
    cudaDeviceGetAttribute(&numberOfSMs, cudaDevAttrWarpSize, deviceId);

    // calculate number of blocks and block size based on hardware properties
    numBlocks = numberOfSMs * 10;
    numThreadsPerBlock = warpSize * 8;

    kernel<<<numBlocks, numThreadsPerBlock>>>();

    // ...
    
}

In most cases, a number of blocks that is a multiple of the number of SMs is recommended.
GPUs are very efficient at switching execution contexts between blocks.
Having multiple blocks available per SM gives the SM the chance to hide latencies occuring from another block, e.g., due to memory accesses.

Assigning the same number of blocks to each SM helps to ensure that each SM has the same amount of work and we don't waste resources by having parts of the GPU idling.

Equally, the block size should usually be a multiple of the warp size for optimal hardware utilization.

## Next Step

To continue with the course, head over to the [reductions](./05-reductions.ipynb) notebook.